# Online Boutique AIOps — 数据集读取与自动化训练/评测 Pipeline

本 notebook 演示如何使用 `online_boutique_rca_full_v1` 数据集：

1. **读取数据集**（train / valid / test 三个划分 + 隐藏的 test 答案）
2. **实现你自己的算法**（只需实现 `fit` 和 `predict` 两个方法）
3. **交给 Pipeline 自动完成**：标准化 → 训练 → 推理 → 评测 → 绘图
4. **查看结果**：所有产物按时间戳存入 `output/<timestamp>_<run_name>/`，不会互相覆盖

---

### 防泄露设计（重要）

- 数据集按**采集时间顺序**划分：train 全部早于 valid，valid 早于 test，无未来信息泄露。
- 标准化统计量（mean/std）**只在 train 上拟合**，pipeline 自动用同一组统计量变换 valid/test。
- **test 的标签不在 `processed/`**，只在 `answers/`，你的算法看不到 test 答案。

## 0. 环境准备

从项目根目录运行本 notebook（`benchmark` 包需在 import 路径上）。

In [ ]:
import sys
from pathlib import Path

# 确保能 import 到项目的 benchmark 包（notebook 在 notebooks/ 下，项目根在上一级）
ROOT = Path.cwd()
if (ROOT / "benchmark").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "benchmark").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("找不到 benchmark 包，请从项目根目录或 notebooks/ 运行")
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from benchmark.pipeline import Pipeline, DatasetBundle, PipelineContext

DATASET = PROJECT_ROOT / "data" / "datasets" / "online_boutique_rca_full_v1"
print("项目根:", PROJECT_ROOT)
print("数据集:", DATASET)

## 1. 读取数据集

`DatasetBundle.load()` 一次性加载所有划分、答案文件、标准化统计量与元信息。

In [ ]:
bundle = DatasetBundle.load(DATASET)

print("数据集:", bundle.meta["dataset_name"])
print("划分策略:", bundle.meta["split_policy"])
print("特征数:", len(bundle.feature_cols))
print()
print(f"train: {len(bundle.train_x):5d} 行 | 异常点 {bundle.meta['train_anomaly_points']:4d} | runs={bundle.meta['train_runs']}")
print(f"valid: {len(bundle.valid_x):5d} 行 | 异常点 {bundle.meta['valid_anomaly_points']:4d} | runs={bundle.meta['valid_runs']}")
print(f"test : {len(bundle.test_x):5d} 行 | 异常点 {bundle.meta['test_anomaly_points']:4d} | runs={bundle.meta['test_runs']}")

In [ ]:
# 检查时序顺序：train 结束 < valid 开始 < test 开始
print("train:", bundle.train_x.timestamp.min(), "->", bundle.train_x.timestamp.max())
print("valid:", bundle.valid_x.timestamp.min(), "->", bundle.valid_x.timestamp.max())
print("test :", bundle.test_x.timestamp.min(),  "->", bundle.test_x.timestamp.max())
assert bundle.train_x.timestamp.max() < bundle.valid_x.timestamp.min() < bundle.test_x.timestamp.min()
print("\n时序顺序正确，无未来信息泄露。")

In [ ]:
# 看一眼特征与标签
display(bundle.train_x.head(3))
display(bundle.train_y.head(3))
print("标签列:", list(bundle.train_y.columns))
print("phase 取值:", bundle.train_y.phase.unique().tolist())

## 2. 可视化数据（可选）

看看故障期间的指标变化。cpu_stress 会抬高 recommendationservice 的 CPU 与延迟，pod_kill 会引发 cartservice 的 restart/error。

In [ ]:
ty = bundle.test_x.copy()
ty["is_anomaly"] = bundle.test_truth["y_true"].values
ts = pd.to_datetime(ty.timestamp)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(ts, ty["recommendationservice_cpu_usage"], lw=0.8)
axes[0].set_ylabel("recommend CPU")
axes[0].set_title("test 集指标（橙色 = 真实异常区间）")
axes[1].plot(ts, ty["cartservice_error_rate"], lw=0.8, color="#d62728")
axes[1].set_ylabel("cart error_rate")
for ax in axes:
    prev = 0; start = None
    for i, v in enumerate(ty.is_anomaly.values):
        if v and not prev: start = ts.iloc[i]
        elif not v and prev: ax.axvspan(start, ts.iloc[i], color="orange", alpha=0.25)
        prev = v
plt.tight_layout(); plt.show()

## 3. 实现你的算法 （⭐ 用户在这里写代码）

实现一个检测器类，只需两个方法：

```python
def fit(self, train_x, train_y, valid_x, valid_y, ctx):
    # 在这里训练。train_x/valid_x 已由 pipeline 用 train-only 统计量标准化。
    # 无监督方法可忽略 *_y。

def predict(self, test_x, ctx) -> np.ndarray:
    # 返回一维异常分数，长度 == len(test_x)，值越大越异常。
```

`ctx` 提供：`ctx.feature_cols`、`ctx.norm_stats`、`ctx.meta`、`ctx.output_dir`、`ctx.scale(df)`。

下面给出一个**示例实现**（高斯概率密度基线），你可以直接替换成自己的模型（Isolation Forest、AutoEncoder、LSTM 等）。

In [ ]:
class MyDetector:
    """示例：在标准化空间拟合对角高斯，用负对数似然作为异常分数。

    这是一个纯无监督基线：只用 train 的正常点估计均值/方差。"""

    def fit(self, train_x, train_y, valid_x, valid_y, ctx):
        self.cols = [c for c in ctx.feature_cols if c in train_x.columns]
        # 只用正常样本拟合（is_anomaly==0），更贴近异常检测设定
        normal_mask = (train_y["is_anomaly"] == 0).values
        X = train_x.loc[normal_mask, self.cols].to_numpy(dtype=float)
        self.mu = X.mean(axis=0)
        self.var = X.var(axis=0) + 1e-6

    def predict(self, test_x, ctx):
        X = test_x[self.cols].to_numpy(dtype=float)
        # 负对数高斯密度 = 0.5 * sum((x-mu)^2/var + log(var))，越大越异常
        score = 0.5 * (((X - self.mu) ** 2) / self.var + np.log(self.var)).sum(axis=1)
        return score

## 4. 运行 Pipeline

`Pipeline.run()` 自动完成：

1. 用 **train-only** 统计量标准化 train/valid/test
2. 调用你的 `fit` / `predict`
3. 对照 `answers/` 评测检测指标（F1 / ROC-AUC / PR-AUC）与逐 incident 检出率
4. 绘图（分数时间线 / ROC-PR / 分数分布）
5. 全部产物写入 `output/<时间戳>_<run_name>/`

In [ ]:
pipe = Pipeline(bundle, run_name="my_detector", output_root=PROJECT_ROOT / "output")
result = pipe.run(MyDetector())

## 5. 查看结果

In [ ]:
import json
print(json.dumps(result.metrics, indent=2, ensure_ascii=False))

In [ ]:
# 展示生成的图表
from IPython.display import Image, display
for name in ["score_timeline.png", "roc_pr_curves.png", "score_distribution.png"]:
    print(name)
    display(Image(str(result.output_dir / name)))

In [ ]:
# 逐 incident 检出情况
per_inc = pd.read_csv(result.output_dir / "per_incident.csv")
print(f"检出 {per_inc.detected.sum()}/{len(per_inc)} 个 incident")
display(per_inc.head(10))

## 6. 反复实验

每次 `pipe.run()` 都会创建一个新的时间戳目录，不会覆盖历史结果。你可以换不同算法/参数多次运行，然后对比 `output/` 下的各次结果。

### 离线提交方式

如果你在别处跑出了预测分数，也可以直接提交一个 `submission` DataFrame（参考 `examples/sample_submission.csv`），跳过 fit/predict：

```python
sub = pd.read_csv(DATASET / "examples" / "sample_submission.csv")
sub["anomaly_score"] = my_scores   # 填入你的分数
result = pipe.run(detector=None, submission=sub)
```

In [ ]:
# 列出 output/ 下所有历史运行
out_root = PROJECT_ROOT / "output"
if out_root.exists():
    for d in sorted(out_root.iterdir()):
        if d.is_dir() and (d / "metrics.json").exists():
            m = json.loads((d / "metrics.json").read_text())
            det = m["detection"]
            print(f"{d.name}: F1={det['f1']:.3f} ROC-AUC={det['roc_auc']:.3f} "
                  f"RCA={m['rca']['incident_detection_rate']*100:.0f}%")